# Main Model

## Overview

The Main Model serves as the orchestrator and central coordinator of the multi-agent system. It interprets user requests, determines the appropriate workflow, and delegates tasks to the Architecture, Builder, and Review models in the correct sequence.

Rather than producing code or performing analysis itself, the Main Model manages the entire development pipeline — routing information, requesting files when needed, and ensuring that each specialized agent receives the right context and tasks. It is the only agent that communicates directly with the user, providing clear, structured responses while maintaining strict control over the multi-step workflow.

In the orchestrator, the Main Model acts as the project manager: it organizes the work, maintains consistency, enforces proper agent communication formats, and ensures that development moves smoothly from initial request to fully reviewed output.

## Example Usage

### System Instruction

In [21]:
from system_instruction import getMainSystemInstruction

system_instructon = getMainSystemInstruction()

In [ ]:

from openai import OpenAI
import json
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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" 
)

# TODO: change the model to deepseek 
def call_agent(system_prompt, user_message, model="openai/gpt-oss-20b:free"):
    """Call any orchestrator agent with system + user messages."""
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    return resp.choices[0].message.content


### Flow 1: User → Main → Architecture → Builder → Reviewer → Main


In [23]:
user_request = "Create a VS Code extension that highlights TODO comments."

main_output_1 = call_agent(
    system_instructon,
    json.dumps({"user_request": user_request})
)

print("Main Agent Output to Architecture:")
print(main_output_1)


Main Agent Output to Architecture:
{
  "target": "architecture",
  "files_requested": [],
  "regex_to_run": [],
  "context": "User wants a VS Code extension that highlights TODO comments. The extension should run in VS Code and apply a custom text editor decoration to any line containing the word 'TODO' so that it is visually highlighted.",
  "tasks": [
    "Design extension manifest (package.json) with necessary contributions and activation events.",
    "Define a language-agnostic selector that triggers on 'TODO' comments.",
    "Specify a text decoration style for highlighting TODO lines.",
    "Plan a simple activation function that registers a TextDocumentContentProvider or a set of decorators.",
    "Ensure the extension is portable across major VS Code platforms."
  ],
  "response": ""
}


### Flow 2: User → Main → User


In [25]:
user_request2 = "How to install npm from cli?"

main_output_2 = call_agent(
    system_instructon,
    json.dumps({"user_request": user_request2})
)

print("Main Agent Output to Architecture:")
print(main_output_2)


Main Agent Output to Architecture:
{"target":"user","response":"To install npm from the command line, you first need to install Node.js, since npm comes bundled with it. Here are the basic steps for different operating systems:\n\n**1. Windows**\n1. Download the Node.js installer from the official site: https://nodejs.org/en/download/\n2. Run the installer and follow the prompts. npm will be installed automatically.\n3. Verify the installation in a new command prompt:\n   ```powershell\n   node -v\n   npm -v\n   ```\n\n**2. macOS (using Homebrew)**\n1. Install Homebrew if you haven’t already:\n   ```bash\n   /bin/bash -c \"$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)\"\n   ```\n2. Install Node.js (npm is included):\n   ```bash\n   brew install node\n   ```\n3. Check versions:\n   ```bash\n   node -v\n   npm -v\n   ```\n\n**3. Linux (Debian/Ubuntu)**\n1. Update package lists:\n   ```bash\n   sudo apt update\n   ```\n2. Install Node.js and npm:\n   ```

In [27]:
user_request3 = """
Is this code correct? 
int main(){
    return a + 7;
}
"""

main_output_3 = call_agent(
    system_instructon,
    json.dumps({"user_request": user_request3})
)

print("Main Agent Output to Architecture:")
print(main_output_3)

Main Agent Output to Architecture:
{
  "target": "architecture",
  "context": "User requests verification of C++ code:\n\nint main(){\n    return a + 7;\n}\n",
  "tasks": ["Analyze the provided code for correctness, identify any errors or undefined behavior, and explain whether it will compile and run as intended."]
}
